In [6]:
#image_path="570216.png"
image_path="1920x1080-full-hd-nature-landscape.jpg"


In [7]:

from PIL import Image
import numpy as np

def image_to_tensor(filepath, dtype=np.float32, normalize=True):
    # Open image and convert to RGB
    image = Image.open(filepath).convert('RGB')  # ensures 3 channels

    # Convert to NumPy array (H, W, C)
    img_np = np.array(image, dtype=dtype)

    match dtype:
        case np.float32:
            img_np /= 255.0  # normalize to [0, 1]
        case np.int32:
            img_np =np.clip(img_np,0,255)

    # Transpose to (C, H, W)
    img_np = np.transpose(img_np, (2, 0, 1))

    # Add batch dimension -> (1, C, H, W)
    img_tensor = np.expand_dims(img_np, axis=0)

    return img_tensor  # shape: [1, C, H, W]




In [8]:
def create_kernel(kernel_size=3):

    channels = 3
    

    # Luminance coefficients for RGB
    luminance = np.array([0.2989, 0.5870, 0.1140], dtype=np.float32)

    # Create empty kernel: shape [1, channels, kernel_size, kernel_size]
    mean_kernel = np.zeros((1, channels, kernel_size, kernel_size), dtype=np.float32)

    # Fill each channel with its luminance spread over the kernel
    for c in range(channels):
        mean_kernel[0, c, :, :] = luminance[c] / (kernel_size ** 2)

    # mean_kernel is now ready to use
    #print("Kernel shape:", mean_kernel.shape)
    #print(mean_kernel)
    np.save("w.npy",mean_kernel)
    mean_kernel_q =  pow(2, 8)*mean_kernel
    mean_kernel_q = mean_kernel_q.astype(np.uint32)
    #print(mean_kernel_q)
    np.save("w_uint32.npy",mean_kernel_q)
    return mean_kernel,mean_kernel_q

In [9]:
import torch
import torch.nn as nn
import torchvision.transforms as T
from PIL import Image
import torch.nn.functional as F


stride = 2
padding = 1  # keep things centered

mean_kernel,mean_kernel_q = create_kernel()
input_torch = torch.from_numpy(image_to_tensor(image_path)).to(torch.float32)
weight_torch = torch.from_numpy(mean_kernel).to(torch.float32)
#Apply convolution
downsampled = F.conv2d(
        input_torch,
        weight_torch,
        bias=None,
        stride=stride,
        padding=padding
    )
# Save output
output = T.ToPILImage()(downsampled.squeeze(0).clamp(0, 1))
output.save("output_downsampled.jpg")

In [10]:
input_torch = torch.from_numpy(image_to_tensor(image_path,dtype=np.int32)).to(torch.int32)
weight_torch = torch.from_numpy(mean_kernel_q).to(torch.int32)
#Apply convolution
downsampled = F.conv2d(
        input_torch,
        weight_torch,
        bias=None,
        stride=stride,
        padding=padding
    )
# Save output
downsampled =downsampled.squeeze(0)/(pow(2, 8))
downsampled =downsampled.clamp(0,255).to(torch.uint8)
output = T.ToPILImage()(downsampled)
output.save("output_downsampled_q.jpg")